# Preprocessing Dataset SUNRGBD
Skrip ini bertugas membongkar struktur folder SUNRGBD yang bersarang (nested) menjadi struktur datar (flat directory). 

Setiap direktori *scene* di dalam SUNRGBD biasanya memiliki sub-folder `image` dan `depth`. Skrip ini akan mencari folder-folder tersebut, menyalin gambar dan peta kedalamannya, lalu mengganti namanya menjadi format yang seragam (contoh: `sunrgbd_000001.jpg`).

In [3]:
import os
import shutil
import scipy.io

def flatten_sunrgbd(src_dir, dst_dir):
    # Membuat folder tujuan
    out_img_dir = os.path.join(dst_dir, 'images')
    out_depth_dir = os.path.join(dst_dir, 'depths')
    out_label_dir = os.path.join(dst_dir, 'labels')

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_depth_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    print(f"Memulai ekstraksi dataset dari: {src_dir}")
    print(f"Tujuan: {dst_dir}\n")

    scene_count = 0
    
    # Menyisir seluruh folder dan sub-folder
    for root, dirs, files in os.walk(src_dir):
        if 'image' in dirs and 'depth' in dirs:
            scene_count += 1
            img_folder = os.path.join(root, 'image')
            depth_folder = os.path.join(root, 'depth')

            # Mengambil file pertama dari masing-masing folder (biasanya hanya ada 1)
            imgs = [f for f in os.listdir(img_folder) if f.endswith('.jpg')]
            depths = [f for f in os.listdir(depth_folder) if f.endswith('.png')]

            if imgs and depths:
                img_name = imgs[0]
                depth_name = depths[0]

                # Penamaan ulang agar terstruktur dan unik
                new_basename = f"sunrgbd_{scene_count:06d}"

                # Salin dan ganti nama file gambar RGB
                shutil.copy(
                    os.path.join(img_folder, img_name),
                    os.path.join(out_img_dir, f"{new_basename}.jpg")
                )
                # Salin dan ganti nama file Depth map
                shutil.copy(
                    os.path.join(depth_folder, depth_name),
                    os.path.join(out_depth_dir, f"{new_basename}.png")
                )

                # [PLACEHOLDER PARSER LABEL .MAT KE .TXT]
                # Ekstraksi label SUNRGBD memerlukan logika konversi proyeksi kamera 3D ke 2D
                # Untuk saat ini, kita buat file .txt kosong agar Dataloader tidak crash 
                # dan bisa menghitung Depth Loss.
                with open(os.path.join(out_label_dir, f"{new_basename}.txt"), 'w') as f:
                    pass 
                    
            if scene_count % 500 == 0:
                print(f"Berhasil memproses {scene_count} scenes...")

    print(f"\nEkstraksi selesai! Total {scene_count} pasang gambar dan depth berhasil diekstrak ke {dst_dir}.")

# =========================================
# BLOK EKSEKUSI
# =========================================
# Sesuaikan path dengan lokasi folder SUNRGBD mentah Anda di VS Code
source_directory = 'datasets/SUNRGBD' 
destination_directory = 'data/sunrgbd_flat'

# Uncomment baris di bawah untuk menjalankan:
flatten_sunrgbd(source_directory, destination_directory)


Memulai ekstraksi dataset dari: datasets/SUNRGBD
Tujuan: data/sunrgbd_flat

Berhasil memproses 500 scenes...
Berhasil memproses 1000 scenes...
Berhasil memproses 1500 scenes...
Berhasil memproses 2000 scenes...

Ekstraksi selesai! Total 2003 pasang gambar dan depth berhasil diekstrak ke data/sunrgbd_flat.
